# 01 · Houston 2025: catching a slow bleed

Nick Chubb started 2025 as Houston's lead back and slowly lost his role to rookie Woody Marks. This is the exact problem RosterWatch is built for.

This notebook walks through the data and the CUSUM slow-bleed detector step by step. The reusable logic lives in `src/`; this notebook only explores it.

Run `python -m src.pipeline` once from the repo root first, so the data is cached.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))   # so `import src` works from notebooks/

import matplotlib.pyplot as plt
from src import data, trend
from src.config import HISTORY

## 1. Weekly usage

One row per player-game. `snap_pct` = share of offensive snaps. `passdown_share` = share of the team's targets + carries on 3rd/4th down or in the last 2 minutes of a half.

In [ ]:
usage = data.load_weekly_usage(HISTORY + [HISTORY[-1] + 1])
hou = usage[(usage.season == 2025) & (usage.team == "HOU") & usage.player.isin(["Nick Chubb", "Woody Marks"])]
hou.pivot(index="week", columns="player", values=["snap_pct", "passdown_share"]).round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharex=True)
for ax, col, title in [(axes[0], "snap_pct", "Snap share"), (axes[1], "passdown_share", "Passing-down share")]:
    for player, g in hou.groupby("player"):
        ax.plot(g.week, g[col], marker="o", label=player)
    ax.set_title(title); ax.set_xlabel("Week"); ax.set_ylim(0, 1)
axes[0].legend()
plt.tight_layout()

## 2. The CUSUM tally

1. **Established level** = average of the player's first 3 games.
2. Each later game gets a z-score: how far it landed from that level, in units of normal game-to-game noise for the position (fitted on 2022–25).
3. **Bleed tally** = `min(0, tally + z + 0.5)`. Games within half an SD of normal don't count, and small shortfalls add up. The flag fires at −3.

In [ ]:
noise = trend.fit_game_noise(usage[usage.season.isin(HISTORY)])
paths = {p: trend.cusum_path(g, noise) for p, g in hou.groupby("player")}

fig, ax = plt.subplots(figsize=(8, 3.5))
for p, c in paths.items():
    ax.plot(c.week, c.cusum_bleed, marker="o", label=f"{p}: bleed")
    ax.plot(c.week, c.cusum_climb, marker="s", linestyle="--", label=f"{p}: climb")
for h in (3, -3):
    ax.axhline(h, color="grey", linewidth=1, linestyle=":")
ax.set_xlabel("Week"); ax.set_ylabel("tally (SD units)"); ax.legend(fontsize=8)
plt.tight_layout()

In [ ]:
cus = trend.cusum_all(usage[usage.season == 2025], noise)
trend.first_crossings(cus, ["Nick Chubb", "Woody Marks"], 2025, h=3)

## Try it yourself

- Change `h=3` to 2 or 4. How much earlier or later do the flags fire? Check `trend.cusum_backtest` for the cost in false alarms.
- Pick another backfield (e.g. `team == "MIN"`, Jordan Mason vs Aaron Jones). Does the backup's climb fire before the starter's bleed again?
- Change `trend.ESTABLISH_GAMES` from 3 to 4. What changes, and why?